<a href="https://colab.research.google.com/github/trludewig/MentalHealthEDA/blob/main/Mental_Health_EDA_Saved_on_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile
import os

# Define the path to the uploaded zip file
zip_file_path = 'archive (5).zip'


# Define the directory where you want to extract the contents
extraction_path = 'uploaded_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

# Extract the zip file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"'{zip_file_path}' extracted to '{extraction_path}'")

# List the contents of the extracted directory to find the CSV file
print("Contents of extracted directory:")
for root, dirs, files in os.walk(extraction_path):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import pandas as pd
import os

# Construct the full path to the extracted CSV file
csv_file_path = os.path.join(extraction_path, 'final_depression_dataset_1.csv')

print(f"Loading CSV file: {csv_file_path}")

# Load the CSV file into a pandas DataFrame
depression_df = pd.read_csv(csv_file_path)

# Display the first 5 rows of the DataFrame
print("\nDataFrame head:")
print(depression_df.head())

# New Section

In [ ]:
!pip install ydata-profiling
from ydata_profiling import ProfileReport

# Generate the profile report for depression_df
profile = ProfileReport(depression_df, title="Depression Dataset Profile Report")

# Display the report
profile.to_notebook_iframe()

Section 1 – Exploratory Data Analysis (Foundational)

1.   How many individuals are in the dataset? What percentage are labeled as depressed?
 2,556 individuals, 18% are depressed.
2.   Identify missing values and data quality issues.
The data quality is pretty good!  Students all have answers for student questions, professionals all have questions about their jobs.
3.  What is the most common profession among depressed individuals?  Teacher
4. Which profession has the highest depression rate?
Graphic Designer
5. How does age and gender impact whether you're depressed?






In [ ]:
working_df = depression_df[depression_df['Working Professional or Student']=='Working Professional']
student_df = depression_df[depression_df['Working Professional or Student']=='Student']
depressed_yes_df = depression_df[depression_df['Depression']=='Yes']
student_df.reset_index(drop=True, inplace=True)
working_df.reset_index(drop=True, inplace=True)
depressed_yes_df.reset_index(drop=True, inplace=True)
student_df

In [ ]:
(depression_df['Depression']=='Yes').mean()*100 # 18% are depressed
# 2054 working professionals and 502 students
# Confirmed
# find out if those without profession are students or if students put Other Value for profession.
# study satisfaction, gpa, - all students have a value for these features, i think
# work pressure, job satisfaction - all workers have a value for these features, i think.

depression_df['Working Professional or Student'].value_counts()
print(depression_df[depression_df['Working Professional or Student']=='Student']['CGPA'].isna().sum())
print(depression_df[depression_df['Working Professional or Student']=='Working Professional']['Job Satisfaction'].isna().sum())

In [ ]:
profession_depression_rate = working_df.groupby('Profession')['Depression'].value_counts(normalize=True).unstack(fill_value=0)
profession_depression_rate.sort_values(by='Yes', ascending=False, inplace=True)
profession_depression_rate

# Here we see that all the Students have no Profession value and much higher rate of depression than those in professional world
test = depression_df.groupby('Profession', dropna=False)['Depression'].value_counts(normalize=True).unstack(fill_value=0)
test

# Students are much more depressed than working professionals...could be age, etc.
depression_df.groupby('Working Professional or Student')['Depression'].value_counts(normalize=True).unstack(fill_value=0)


In [ ]:
#What is the most common profession among depressed individuals?

depressed_yes_df['Profession'].value_counts(dropna=False) # this tells us the max profession count in the depressed group.
# but this seems like it could be skewed - what if there are 2000 teachers who are not depressed and only 2 hr managers who are not depressed?
# also the 309 are students.

In [ ]:
# How does age and gender impact whether you're depressed?
# Let's look at age plotted with color depressed or not.

#

# How does age and gender impact whether you're depressed?

In this dataset, the younger you are, the more likely you are to be depressed.  Gender is less of a factor.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# How does age correlate with depression?

# Define age bins and labels
age_bins = range(0, depression_df['Age'].max() + 10, 10)
age_labels = [f'{i}-{i+9}' for i in age_bins[:-1]]

# Create a new column 'Age_Group' by binning the 'Age' column
depression_df['Age_Group'] = pd.cut(depression_df['Age'], bins=age_bins, labels=age_labels, right=False)

# Calculate the percentage of depression per age group
depression_by_age_group = depression_df.groupby('Age_Group')['Depression'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
depression_by_age_group.rename(columns={'Depression': 'Depression_Percentage'}, inplace=True)

# Plot the bar chart
plt.figure(figsize=(12, 6))
sns.barplot(
    data=depression_by_age_group,
    x='Age_Group',
    y='Depression_Percentage',
    palette='viridis'
)

plt.title('Depression Percentage by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Depression Percentage (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
#How does gender correlate with depression?

import matplotlib.pyplot as plt
import seaborn as sns

# How does age correlate with depression?

# Calculate the percentage of depression per age group
depression_by_gender = depression_df.groupby('Gender')['Depression'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
depression_by_gender.rename(columns={'Depression': 'Depression_Percentage'}, inplace=True)

# Plot the bar chart
plt.figure(figsize=(12, 6))
sns.barplot(
    data=depression_by_gender,
    x='Gender',
    y='Depression_Percentage',
    palette='viridis'
)

plt.title('Gender Percentage by Age Group')
plt.xlabel('Gender')
plt.ylabel('Depression Percentage (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Task
Calculate and visualize the depression rates by 'Age_Group' and 'Gender' from the `depression_df` DataFrame using a grouped bar chart.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

depression_by_age_gender = depression_df.groupby(['Age_Group', 'Gender'], observed=False)['Depression'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
depression_by_age_gender.rename(columns={'Depression': 'Depression_Percentage'}, inplace=True)

# Plotting the grouped bar chart
plt.figure(figsize=(14, 7))
sns.barplot(
    data=depression_by_age_gender,
    x='Age_Group',
    y='Depression_Percentage',
    hue='Gender',
    palette='tab10' # Using a different palette for gender differentiation
)

plt.title('Depression Percentage by Age Group and Gender')
plt.xlabel('Age Group')
plt.ylabel('Depression Percentage (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='Gender')
plt.tight_layout()
plt.show()

# Task
Task: Preprocess the `depression_df` DataFrame by handling missing values in relevant columns for both 'Working Professional' and 'Student' groups, encoding categorical features using one-hot encoding, and scaling numerical features to prepare the data for logistic regression.

## Data Preprocessing for Logistic Regression

### Subtask:
Handle missing values, encode categorical features, and scale numerical features in the `depression_df` to prepare it for logistic regression.


**Reasoning**:
The subtask requires handling missing values, encoding categorical features, and scaling numerical features to prepare the data for logistic regression. This step involves creating a copy of the DataFrame, filling NaN values, encoding the target variable, identifying feature types, scaling numerical features using StandardScaler, and one-hot encoding categorical features using pd.get_dummies, and finally dropping the 'Name' column, all as specified in the instructions.



In [ ]:
depression_df

In [ ]:
depression_df.info()

i have a mental health dataframe with a row per person and a target label of Depression=Yes/No. The features include professional_or_student, profession, student_gpa, student_stress, work_stress. For students, the work_stress and profession features are all blank. so to prepare the dataset for logistic regression, when i fill na's, all students get 0 for work_stress and profession and this creates some colinnearity, I think. similarly, after filling out the na's, all rows where they have a job have 0 for student_gpa and student_stress. How should i handle this? 1. Should i only look at features that are common to all rows first and remove these other features at first? 2. Then i can do a separate logistic regression on only students and another one on only people who work? What is the best practice in this kind of situation?

The answer from AI is to keep the professional_or_student indicator and the model then figures out that there's an interaction and only when professional_or_student==student then there are certain other features that matter...same for professionals.  Alternatively, you CAN split into several models...students only, professionals only.

In [ ]:
# TODO move the logistic regression preprocessing here and then check for multi-collinearity below.

# Task
Calculate the Variance Inflation Factor (VIF) for the numerical features in the `df_processed` DataFrame to identify and address any significant multicollinearity.

## Check for Multicollinearity

### Subtask:
Calculate the Variance Inflation Factor (VIF) for the numerical features in the `df_processed` DataFrame to identify and address any significant multicollinearity that could affect model stability and interpretation.


**Reasoning**:
To check for multicollinearity, I will import the necessary function, extract the numerical features, calculate VIF for each, and display the results.



## Verify Linearity of Continuous Predictors with Log Odds

### Subtask:
For each continuous numerical feature, visualize its relationship with the log odds of the 'Depression' outcome to assess the assumption of linearity. This involves binning the continuous variable and plotting the mean of the binned variable against the log odds of depression.


**Reasoning**:
To verify the linearity assumption of continuous predictors with log odds, I will iterate through each numerical feature, bin the data into 10 groups, calculate the mean of the feature and the proportion of depression for each bin, convert these proportions to log odds, and then visualize the relationship using a scatter plot for each feature. I will handle edge cases where the depression probability is 0 or 1 by clipping the values to avoid errors in log odds calculation.



## Summary:

### Data Analysis Key Findings

*   **Multicollinearity Identified:** VIF scores revealed multicollinearity among several numerical features.
    *   'CGPA' had a VIF of 7.68, indicating significant multicollinearity.
    *   'Study Satisfaction' (VIF: 4.18) and 'Academic Pressure' (VIF: 4.13) also showed notable multicollinearity.
    *   Other numerical features had lower VIF values.
*   **Linearity Assumption Visualized:** Plots were successfully generated for each continuous numerical feature ('Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Work/Study Hours', 'Financial Stress') against the log-odds of 'Depression'. These visualizations are intended to assess the linearity assumption of the relationship.

### Insights or Next Steps

*   **Address Multicollinearity:** Given the high VIF scores, particularly for 'CGPA', 'Study Satisfaction', and 'Academic Pressure', it is crucial to address multicollinearity before model building. This could involve removing one of the highly correlated variables, combining them into an index, or using techniques like principal component analysis.
*   **Evaluate Linearity Plots:** Carefully examine the generated log-odds plots for each feature. If any plot shows a non-linear relationship, consider transforming the respective feature (e.g., log transformation, polynomial terms) or using non-linear modeling approaches to better capture the relationship with the dependent variable.


# Task
Build and evaluate a logistic regression model on the preprocessed data, including the extraction of model coefficients and p-values.

## Build and Train Logistic Regression Model

### Subtask:
Split the preprocessed data into training and testing sets, then train a logistic regression model using the training data.


Do a test with a similar little fake dataset to see how we avoid the single error i keep getting
LinAlgError: Singular matrix having to do with messed up collinearity

In [ ]:
# This example works...let's copy this.
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ── 0. Create fake dataset (non-zero values guaranteed per group) ──────────────
np.random.seed(42)
n = 500

students = pd.DataFrame({
    'id': np.arange(n // 2, n),
    'student_or_professional': 'student',
    'profession':    np.nan,
    'CGPA':          np.random.uniform(5.0, 10.0, n // 2),   # always non-zero
    'work_stress':   np.nan,
    'school_stress': np.random.uniform(1, 10, n // 2),        # always non-zero
    'work_satisfaction':   np.nan,
    'school_satisfaction': np.random.uniform(1, 10, n // 2),        # always non-zero
    'depression':    np.random.choice(['Yes', 'No'], n // 2, p=[0.4, 0.6])
})

professionals = pd.DataFrame({
    'id': np.arange(n // 2, n),
    'student_or_professional': 'professional',
    'profession':    np.random.choice(['doctor', 'nurse', 'teacher'], n // 2),
    'CGPA':          np.nan,
    'work_stress':   np.random.uniform(1, 10, n // 2),         # always non-zero
    'school_stress': np.nan,
    'work_satisfaction':   np.random.uniform(1, 10, n // 2),         # always non-zero
    'school_satisfaction': np.nan,
    'depression':    np.random.choice(['Yes', 'No'], n // 2, p=[0.3, 0.7])
})

df = pd.concat([students, professionals], ignore_index=True).sample(frac=1, random_state=42)
print("Raw dataframe sample:")
print(df.head(8).to_string())

# ── 1. Encode the target variable ──────────────────────────────────────────────
df['depression'] = df['depression'].map({'Yes': 1, 'No': 0})

# ── 2. Fill profession nulls → encode → drop student_or_professional ───────────
df['profession'] = df['profession'].fillna('student')
df = df.drop(columns=['student_or_professional'])
df = pd.get_dummies(df, columns=['profession'], drop_first=True)

# ── 3. Fill NaNs with group mean instead of 0 ─────────────────────────────────
# Filling with 0 would cause perfect collinearity because:
#   CGPA == 0       perfectly predicts profession_student == 0 (professional)
#   work_stress == 0 perfectly predicts profession_student == 1 (student)
# Using group mean breaks this perfect signal while still being plausible.
cgpa_mean         = df.loc[df['profession_student'] == 1, 'CGPA'].mean()
work_stress_mean  = df.loc[df['profession_student'] == 0, 'work_stress'].mean()
school_stress_mean= df.loc[df['profession_student'] == 1, 'school_stress'].mean()

df['CGPA']          = df['CGPA'].fillna(0)
df['work_stress']   = df['work_stress'].fillna(0)
df['school_stress'] = df['school_stress'].fillna(0)
df['work_satisfaction']   = df['work_satisfaction'].fillna(0)
df['school_satisfaction'] = df['school_satisfaction'].fillna(0)
df = df.drop(columns=['id'])

print("Final dataframe sample:")
print(df.head(8).to_string())

# ── 4. Split into features (X) and target (y) ──────────────────────────────────
X = df.drop(columns=['depression'])
y = df['depression']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 5. Scale numeric features ──────────────────────────────────────────────────
# IMPORTANT: compute group means on train set only to avoid data leakage
numeric_cols = ['CGPA', 'work_stress', 'school_stress', 'work_satisfaction', 'school_satisfaction']
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

# ── 6. Train logistic regression ───────────────────────────────────────────────
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# ── 7. Evaluate ────────────────────────────────────────────────────────────────
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print("\n── Confusion Matrix ──────────────────────────────")
print(confusion_matrix(y_test, y_pred))

print("\n── Classification Report ─────────────────────────")
print(classification_report(y_test, y_pred))

print("── ROC-AUC Score ─────────────────────────────────")
print(f"  {roc_auc_score(y_test, y_pred_prob):.4f}")

print("\n── Model Coefficients ────────────────────────────")
coef_df = pd.DataFrame({
    'Feature':     X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))

In [ ]:
# Try using the sample code but switch to statsmodel so i can get p-values
# Works and prints p-values
import statsmodels.api as sm
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ── 0. Create fake dataset (non-zero values guaranteed per group) ──────────────
np.random.seed(42)
n = 500

students = pd.DataFrame({
    'id': np.arange(n // 2, n),
    'student_or_professional': 'student',
    'profession':    np.nan,
    'CGPA':          np.random.uniform(5.0, 10.0, n // 2),   # always non-zero
    'work_stress':   np.nan,
    'school_stress': np.random.uniform(1, 10, n // 2),        # always non-zero
    'work_satisfaction':   np.nan,
    'school_satisfaction': np.random.uniform(1, 10, n // 2),        # always non-zero
    'depression':    np.random.choice(['Yes', 'No'], n // 2, p=[0.4, 0.6])
})

professionals = pd.DataFrame({
    'id': np.arange(n // 2, n),
    'student_or_professional': 'professional',
    'profession':    np.random.choice(['doctor', 'nurse', 'teacher'], n // 2),
    'CGPA':          np.nan,
    'work_stress':   np.random.uniform(1, 10, n // 2),         # always non-zero
    'school_stress': np.nan,
    'work_satisfaction':   np.random.uniform(1, 10, n // 2),         # always non-zero
    'school_satisfaction': np.nan,
    'depression':    np.random.choice(['Yes', 'No'], n // 2, p=[0.3, 0.7])
})

df = pd.concat([students, professionals], ignore_index=True).sample(frac=1, random_state=42)
print("Raw dataframe sample:")
print(df.head(8).to_string())

# ── 1. Encode the target variable ──────────────────────────────────────────────
df['depression'] = df['depression'].map({'Yes': 1, 'No': 0})

# ── 2. Fill profession nulls → encode → drop student_or_professional ───────────
df['profession'] = df['profession'].fillna('student')
df = df.drop(columns=['student_or_professional'])
df = pd.get_dummies(df, columns=['profession'], drop_first=True, dtype=int) # Added dtype=int

# ── 3. Fill NaNs with group mean instead of 0 ─────────────────────────────────
# Filling with 0 would cause perfect collinearity because:
#   CGPA == 0       perfectly predicts profession_student == 0 (professional)
#   work_stress == 0 perfectly predicts profession_student == 1 (student)
# Using group mean breaks this perfect signal while still being plausible.
cgpa_mean         = df.loc[df['profession_student'] == 1, 'CGPA'].mean()
work_stress_mean  = df.loc[df['profession_student'] == 0, 'work_stress'].mean()
school_stress_mean= df.loc[df['profession_student'] == 1, 'school_stress'].mean()

df['CGPA']          = df['CGPA'].fillna(0)
df['work_stress']   = df['work_stress'].fillna(0)
df['school_stress'] = df['school_stress'].fillna(0)
df['work_satisfaction']   = df['work_satisfaction'].fillna(0)
df['school_satisfaction'] = df['school_satisfaction'].fillna(0)
df = df.drop(columns=['id'])

print("Final dataframe sample:")
print(df.head(8).to_string())

# ── 4. Split into features (X) and target (y) ──────────────────────────────────
X = df.drop(columns=['depression'])
y = df['depression']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 5. Scale numeric features ──────────────────────────────────────────────────
# IMPORTANT: compute group means on train set only to avoid data leakage
numeric_cols = ['CGPA', 'work_stress', 'school_stress', 'work_satisfaction', 'school_satisfaction']
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

# ── 6. Train logistic regression using statsmodels ───────────────────────────
# Add the intercept (constant) to both train and test sets
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

# Fit the Logit model
# Note: statsmodels uses (y, X) order unlike sklearn's (X, y)
sm_model = sm.Logit(y_train, X_train_sm).fit(disp=0)

# ── 7. Evaluate ────────────────────────────────────────────────────────────────
# statsmodels predict() returns probabilities directly
y_pred_prob = sm_model.predict(X_test_sm)
y_pred      = (y_pred_prob > 0.5).astype(int)

print("\n── Confusion Matrix ──────────────────────────────")
print(confusion_matrix(y_test, y_pred))

print("\n── Classification Report ─────────────────────────")
print(classification_report(y_test, y_pred))

print(f"── ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob):.4f} ──")

print("\n── Model Coefficients & P-Values (Sorted Descending) ──")
# Create a DataFrame for the summary stats
results_df = pd.DataFrame({
    'Feature': sm_model.params.index,
    'Coefficient': sm_model.params.values,
    'P-value': sm_model.pvalues.values
})

# Sort by Coefficient descending
results_df = results_df.sort_values(by='Coefficient', ascending=False)

# Print with formatting for readability
print(results_df.to_string(index=False, formatters={'Coefficient': '{:.4f}'.format, 'P-value': '{:.4f}'.format}))

In [ ]:
depression_df.columns.to_list()

In [ ]:
# WORKS BUT DOES NOT PRINT P-VALUES better to use statsmodel not sklearn
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

df = depression_df.copy()
print("Raw dataframe sample:")
print(df.head(8).to_string())

numeric_cols = [
    'Age', 'Work/Study Hours', 'Financial Stress',
    'Academic Pressure', 'Work Pressure', 'CGPA',
    'Study Satisfaction', 'Job Satisfaction'
]

categorical_features = [
   'Profession', 'Gender', 'City', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness'
]


# ── 1. Encode the target variable ──────────────────────────────────────────────
df['Depression'] = df['Depression'].map({'Yes': 1, 'No': 0})

# ── 2. Fill profession nulls → encode → drop student_or_professional ───────────
df['Profession'] = df['Profession'].fillna('Student')
df = df.drop(columns=['Working Professional or Student', 'Name'])
df = pd.get_dummies(df, columns=categorical_features, drop_first=True)

df['CGPA']          = df['CGPA'].fillna(0)
df['Work Pressure']   = df['Work Pressure'].fillna(0)
df['Academic Pressure'] = df['Academic Pressure'].fillna(0)
df['Job Satisfaction']   = df['Job Satisfaction'].fillna(0)
df['Study Satisfaction'] = df['Study Satisfaction'].fillna(0)

print("Final dataframe sample:")
print(df.head(8).to_string())

# ── 4. Split into features (X) and target (y) ──────────────────────────────────
X = df.drop(columns=['Depression'])
y = df['Depression']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 5. Scale numeric features ──────────────────────────────────────────────────
# IMPORTANT: compute group means on train set only to avoid data leakage

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

# ── 6. Train logistic regression ───────────────────────────────────────────────
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# ── 7. Evaluate ────────────────────────────────────────────────────────────────
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print("\n── Confusion Matrix ──────────────────────────────")
print(confusion_matrix(y_test, y_pred))

print("\n── Classification Report ─────────────────────────")
print(classification_report(y_test, y_pred))

print("── ROC-AUC Score ─────────────────────────────────")
print(f"  {roc_auc_score(y_test, y_pred_prob):.4f}")

print("\n── Model Coefficients ────────────────────────────")
coef_df = pd.DataFrame({
    'Feature':     X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))


In [ ]:
depression_df.columns.to_list()

In [ ]:
#DOES NOT WORK DUE TO COLLINEARITY - I tried and tried and couldn't get this tow ork.
#Change to use statsmodel to print out p-values
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

df = depression_df.copy()
print("Raw dataframe sample:")
print(df.head(8).to_string())

numeric_cols = [
    'Age', 'Work/Study Hours', 'Financial Stress',
    'Academic Pressure', 'Work Pressure', 'CGPA',
    'Study Satisfaction', 'Job Satisfaction'
]

categorical_features = [
   'Profession', 'Gender', 'City', # Removed 'Working Professional or Student' from here
   'Sleep Duration', 'Dietary Habits', 'Degree',
   'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness'
]


# ── 1. Encode the target variable ──────────────────────────────────────────────
df['Depression'] = df['Depression'].map({'Yes': 1, 'No': 0})

# ── 2. Handle structural NaNs by filling with group means ───────────────────────
# Calculate means for relevant groups BEFORE any filling that might alter group definitions
student_academic_pressure_mean = df[df['Working Professional or Student'] == 'Student']['Academic Pressure'].mean()
student_cgpa_mean = df[df['Working Professional or Student'] == 'Student']['CGPA'].mean()
student_study_satisfaction_mean = df[df['Working Professional or Student'] == 'Student']['Study Satisfaction'].mean()

professional_work_pressure_mean = df[df['Working Professional or Student'] == 'Working Professional']['Work Pressure'].mean()
professional_job_satisfaction_mean = df[df['Working Professional or Student'] == 'Working Professional']['Job Satisfaction'].mean()

# Fill NaNs in student-specific columns with student means (for professionals, where these are NaN)
df.loc[df['Working Professional or Student'] == 'Working Professional', 'Academic Pressure'] = df.loc[df['Working Professional or Student'] == 'Working Professional', 'Academic Pressure'].fillna(student_academic_pressure_mean)
df.loc[df['Working Professional or Student'] == 'Working Professional', 'CGPA'] = df.loc[df['Working Professional or Student'] == 'Working Professional', 'CGPA'].fillna(student_cgpa_mean)
df.loc[df['Working Professional or Student'] == 'Working Professional', 'Study Satisfaction'] = df.loc[df['Working Professional or Student'] == 'Working Professional', 'Study Satisfaction'].fillna(student_study_satisfaction_mean)

# Fill NaNs in professional-specific columns with professional means (for students, where these are NaN)
df.loc[df['Working Professional or Student'] == 'Student', 'Work Pressure'] = df.loc[df['Working Professional or Student'] == 'Student', 'Work Pressure'].fillna(professional_work_pressure_mean)
df.loc[df['Working Professional or Student'] == 'Student', 'Job Satisfaction'] = df.loc[df['Working Professional or Student'] == 'Student', 'Job Satisfaction'].fillna(professional_job_satisfaction_mean)

# Handle 'Profession' NaN values, filling with 'Unknown' for consistency (though it should be rare after explicit student/professional split)
df['Profession'] = df['Profession'].fillna('Unknown')

# Drop 'Name' and 'Working Professional or Student' columns as they are either not features
# or their information is now implicitly handled by the zero-filled numerical features.
df = df.drop(columns=['Name', 'Working Professional or Student'])

df = pd.get_dummies(df, columns=categorical_features, drop_first=True, dtype=int)

print("Final dataframe sample:")
print(df.head(8).to_string())

# ── 3. Split into features (X) and target (y) ──────────────────────────────────
X = df.drop(columns=['Depression'])
y = df['Depression']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 4. Scale numeric features ──────────────────────────────────────────────────
# IMPORTANT: compute group means on train set only to avoid data leakage

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

# ── 5. Train logistic regression using statsmodels ───────────────────────────
# Add the intercept (constant) to both train and test sets
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

# Fit the Logit model
# Note: statsmodels uses (y, X) order unlike sklearn's (X, y)
sm_model = sm.Logit(y_train, X_train_sm).fit_regularized(disp=0) # Use fit_regularized for robustness

# ── 6. Evaluate ────────────────────────────────────────────────────────────────
# statsmodels predict() returns probabilities directly
y_pred_prob = sm_model.predict(X_test_sm)
y_pred      = (y_pred_prob > 0.5).astype(int)

print("\n── Confusion Matrix ──────────────────────────────")
print(confusion_matrix(y_test, y_pred))

print("\n── Classification Report ─────────────────────────")
print(classification_report(y_test, y_pred))

print(f"── ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob):.4f} ──")

print("\n── Model Coefficients & P-Values (Sorted Descending) ──")
# Create a DataFrame for the summary stats
results_df = pd.DataFrame({
    'Feature': sm_model.params.index,
    'Coefficient': sm_model.params.values,
    'P-value': sm_model.pvalues.values
})

# Sort by Coefficient descending
results_df = results_df.sort_values(by='Coefficient', ascending=False)

# Print with formatting for readability
print(results_df.to_string(index=False, formatters={'Coefficient': '{:.4f}'.format, 'P-value': '{:.4f}'.format}))

In [ ]:
# I will split the models out.  First let's just look at the features that are not specific to students or specific to workers.
# WORKS!!!  But concerns about independent variables.
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

df = depression_df.copy()
print("Raw dataframe sample:")
print(df.head(8).to_string())

numeric_cols = [
    'Age', 'Work/Study Hours', 'Financial Stress',

]

categorical_features = [
   'Profession', 'Gender', 'City', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness'
]


# ── 1. Encode the target variable ──────────────────────────────────────────────
df['Depression'] = df['Depression'].map({'Yes': 1, 'No': 0})

# ── 2. Fill profession nulls → encode → drop student_or_professional ───────────
df['Profession'] = df['Profession'].fillna('Student')
df = df.drop(columns=['Working Professional or Student',
                      'Name', 'Academic Pressure', 'Work Pressure', 'CGPA',
                      'Study Satisfaction', 'Job Satisfaction'])
df = pd.get_dummies(df, columns=categorical_features, drop_first=True, dtype=int)

print("Final dataframe sample:")
print(df.head(8).to_string())

# ── 4. Split into features (X) and target (y) ──────────────────────────────────
X = df.drop(columns=['Depression'])
y = df['Depression']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 5. Scale numeric features ──────────────────────────────────────────────────
# IMPORTANT: compute group means on train set only to avoid data leakage

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])


# ── 6. Train logistic regression using statsmodels ───────────────────────────
# Add the intercept (constant) to both train and test sets
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

# Fit the Logit model
# Note: statsmodels uses (y, X) order unlike sklearn's (X, y)
sm_model = sm.Logit(y_train, X_train_sm).fit(disp=0)

# ── 7. Evaluate ────────────────────────────────────────────────────────────────
# statsmodels predict() returns probabilities directly
y_pred_prob = sm_model.predict(X_test_sm)
y_pred      = (y_pred_prob > 0.5).astype(int)

print("\n── Confusion Matrix ──────────────────────────────")
print(confusion_matrix(y_test, y_pred))

print("\n── Classification Report ─────────────────────────")
print(classification_report(y_test, y_pred))

print(f"── ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob):.4f} ──")

print("\n── Model Coefficients & P-Values (Sorted Descending) ──")
# Create a DataFrame for the summary stats
results_df = pd.DataFrame({
    'Feature': sm_model.params.index,
    'Coefficient': sm_model.params.values,
    'P-value': sm_model.pvalues.values
})

# Sort by Coefficient descending
results_df = results_df.sort_values(by='P-value', ascending=True)

# Print with formatting for readability
print(results_df.to_string(index=False, formatters={'Coefficient': '{:.4f}'.format, 'P-value': '{:.4f}'.format}))


In [ ]:
# IMPROVEMENTS to the above...
# Drop Suicidal thoughts as it could be a symptom of target variable, not cause.
# Group cities based on size and degrees based on industry domain.
# Put cities with < 50 observations into "Other", same with degree.

# For now, i'll remove city and degree altogether.
# I will split the models out.  First let's just look at the features that are not specific to students or specific to workers.
# WORKS!!!  But concerns about independent variables.
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

df = depression_df.copy()
print("Raw dataframe sample:")
print(df.head(8).to_string())

numeric_cols = [
    'Age', 'Work/Study Hours', 'Financial Stress'
]

categorical_features = [
   'Working Professional or Student', 'Gender',  'Sleep Duration', 'Dietary Habits', 'Family History of Mental Illness'
]
#'Have you ever had suicidal thoughts ?','Degree','City', 'Profession'


# ── 1. Encode the target variable ──────────────────────────────────────────────
df['Depression'] = df['Depression'].map({'Yes': 1, 'No': 0})

# ── 2. Fill profession nulls → encode → drop student_or_professional ───────────
#df['Profession'] = df['Profession'].fillna('Student')
#'Working Professional or Student',
df = df.drop(columns=['Profession',
                      'Name', 'Academic Pressure', 'Work Pressure', 'CGPA',
                      'Study Satisfaction', 'Job Satisfaction',
                      'City', 'Degree', 'Have you ever had suicidal thoughts ?'])
df = pd.get_dummies(df, columns=categorical_features, drop_first=True, dtype=int)

print("Final dataframe sample:")
print(df.head(8).to_string())

# ── 4. Split into features (X) and target (y) ──────────────────────────────────
X = df.drop(columns=['Depression'])
y = df['Depression']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 5. Scale numeric features ──────────────────────────────────────────────────
# IMPORTANT: compute group means on train set only to avoid data leakage

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])


# ── 6. Train logistic regression using statsmodels ───────────────────────────
# Add the intercept (constant) to both train and test sets
X_train_sm = sm.add_constant(X_train, has_constant='add') # Explicitly add constant
X_test_sm = sm.add_constant(X_test, has_constant='add') # Explicitly add constant

# Fit the Logit model
# Note: statsmodels uses (y, X) order unlike sklearn's (X, y)
sm_model = sm.Logit(y_train, X_train_sm).fit(disp=0)

# ── 7. Evaluate ────────────────────────────────────────────────────────────────
# statsmodels predict() returns probabilities directly
y_pred_prob = sm_model.predict(X_test_sm)
y_pred      = (y_pred_prob > 0.5).astype(int)

print("\n── Confusion Matrix ──────────────────────────────")
print(confusion_matrix(y_test, y_pred))

print("\n── Classification Report ─────────────────────────")
print(classification_report(y_test, y_pred))

print(f"── ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob):.4f} ──")

print("\n── Model Coefficients & P-Values (Sorted Descending) ──")
# Create a DataFrame for the summary stats
results_df = pd.DataFrame({
    'Feature': sm_model.params.index,
    'Coefficient': sm_model.params.values,
    'P-value': sm_model.pvalues.values
})

# Sort by Coefficient descending
results_df = results_df.sort_values(by='Coefficient', ascending=False)

# Print with formatting for readability
print(results_df.to_string(index=False, formatters={'Coefficient': '{:.4f}'.format, 'P-value': '{:.4f}'.format}))

# Conclusions from the above large dataset with professionals and students.

The younger subjects show more depression.  Age is the biggest factor.
Next most associated with depression are two equally important factors: if you have unhealthy dietary habits or if you are a student.
Next there is a positive correlation between financial stress and depression.
Next there is a negative correlation between sleeping more than 8 hours and depression,

In [ ]:
# NOW let's look only at students and let's add back student-specific features.
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

df = student_df.copy()
print("Raw dataframe sample:")
print(df.head(8).to_string())

numeric_cols = [
    'Age', 'Work/Study Hours', 'Financial Stress', 'Academic Pressure', 'CGPA',
    'Study Satisfaction'
]

categorical_features = [
   'Working Professional or Student', 'Gender',  'Sleep Duration', 'Dietary Habits', 'Family History of Mental Illness'
]
#'Have you ever had suicidal thoughts ?','Degree','City', 'Profession'


# ── 1. Encode the target variable ──────────────────────────────────────────────
df['Depression'] = df['Depression'].map({'Yes': 1, 'No': 0})

# ── 2. Fill profession nulls → encode → drop student_or_professional ───────────
#df['Profession'] = df['Profession'].fillna('Student')
#'Working Professional or Student',
df = df.drop(columns=['Profession',
                      'Name', 'Work Pressure',
                      'Job Satisfaction',
                      'City', 'Degree', 'Have you ever had suicidal thoughts ?'])
df = pd.get_dummies(df, columns=categorical_features, drop_first=True, dtype=int)

print("Final dataframe sample:")
print(df.head(8).to_string())

# ── 4. Split into features (X) and target (y) ──────────────────────────────────
X = df.drop(columns=['Depression'])
y = df['Depression']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 5. Scale numeric features ──────────────────────────────────────────────────
# IMPORTANT: compute group means on train set only to avoid data leakage

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])


# ── 6. Train logistic regression using statsmodels ───────────────────────────
# Add the intercept (constant) to both train and test sets
X_train_sm = sm.add_constant(X_train, has_constant='add') # Explicitly add constant
X_test_sm = sm.add_constant(X_test, has_constant='add') # Explicitly add constant

# Fit the Logit model
# Note: statsmodels uses (y, X) order unlike sklearn's (X, y)
sm_model = sm.Logit(y_train, X_train_sm).fit(disp=0)

# ── 7. Evaluate ────────────────────────────────────────────────────────────────
# statsmodels predict() returns probabilities directly
y_pred_prob = sm_model.predict(X_test_sm)
y_pred      = (y_pred_prob > 0.5).astype(int)

print("\n── Confusion Matrix ──────────────────────────────")
print(confusion_matrix(y_test, y_pred))

print("\n── Classification Report ─────────────────────────")
print(classification_report(y_test, y_pred))

print(f"── ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob):.4f} ──")

print("\n── Model Coefficients & P-Values (Sorted Descending) ──")
# Create a DataFrame for the summary stats
results_df = pd.DataFrame({
    'Feature': sm_model.params.index,
    'Coefficient': sm_model.params.values,
    'P-value': sm_model.pvalues.values
})

# Sort by Coefficient descending
results_df = results_df.sort_values(by='Coefficient', ascending=False)

# Print with formatting for readability
print(results_df.to_string(index=False, formatters={'Coefficient': '{:.4f}'.format, 'P-value': '{:.4f}'.format}))